# 01 Extract, Transform, Load

**Purpose:** ingest the supplied BNPL tables and ABS income data, then build the curated tables in the diagram.

| | Details |
|---|---|
| Inputs | Supplied files in `tables/landing/`; four official ABS 2021 downloads below. |
| Outputs | Cleaned parquets in `tables/raw/` and joined parquets `tables/curated/`; ABS source extracts in `tables/landing/abs_2021/`. |
| Run | Run top to bottom in `mast30034`, from the repository or `code/`. Requires PySpark + Java, pandas, PyArrow, requests and openpyxl. |
| Behaviour | Rebuilds named `raw`/`curated` outputs using overwrite; reuses existing ABS landing extracts. Original supplied `landing` files are untouched. |
| Draft choices | Left joins retain orders; fraud scores match on entity + date; income joined using state+postcode $\leftrightarrow$ SA2 weighted mapping. |

All transaction processing and joins use **PySpark**. pandas only reads the much smaller ABS CSV/Excel sources, which Spark cannot all read natively. All saved data tables are **Parquet**.

Subsequent reruns will **overwrite** the files on disk.

## 1. Paths, Spark and small helpers

Use the existing landing directory and separate output directories. 

`require_key` is a function that enforces one non-null row per join key, so lookup joins cannot duplicate orders. 
- e.g. `User` table cannot contain two rows for the same `user_id` or `consumer_id`

In [19]:
from pathlib import Path
from functools import reduce
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import pandas as pd
import requests
from pyspark.sql import SparkSession, functions as F

# Find the repository from either its root or a nested notebook directory
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tables/landing").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or code/ directory.")
LANDING, RAW, CURATED = [ROOT / "tables" / layer for layer in ["landing", "raw", "curated"]]
ABS = LANDING / "abs_2021"
PART1 = LANDING / "project-2-bnpl-tables-part1"

# Use four local partitions; shuffle partitions control parallel work during joins and aggregations
spark = (SparkSession.builder.master("local[4]").appName("BNPL ETL")
         .config("spark.sql.shuffle.partitions", "32").config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.session.timeZone", "Australia/Melbourne").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def write_table(df, folder, partition=None):
    """Save a Spark DataFrame as Parquet, replacing the target table; optionally partition by a column."""
    # Monthly partitions keep transaction outputs manageable without a file per day
    writer = df.write.mode("overwrite").option("compression", "uncompressed")
    (writer.partitionBy(partition) if partition else writer).parquet(str(folder))

def require_key(df, keys, label):
    """Raise an error if any key is null or repeated, including for keys spanning several columns."""
    # Null keys cannot match; repeated keys on a lookup table multiply joined rows
    null_key = reduce(lambda a, b: a | b, [F.col(k).isNull() for k in keys])

    # Combine column checks with OR: a row is invalid if any part of its key is null
    if df.filter(null_key).limit(1).count() or df.groupBy(*keys).count().filter("count > 1").limit(1).count():
        raise ValueError(f"{label}: null or repeated key {keys}; review before joining.")

## 2. Landing: acquire ABS 2021 income and geographic lookups

We will use the **G02 `Median_tot_prsnl_inc_weekly`** ABS dataset for median weekly personal income in dollars at SA2 level. 

To map `postcode` (from consumer table) to `SA2` areas, we will use the smallest ABS area unit, **mesh blocks**. Decompose both Postal Area and Statistical Area 2 into mesh blocks and find their overlap to calculate a population-weighted average for district median income per consumer, based on their postcode.

- [2021 Census DataPacks](https://www.abs.gov.au/census/find-census-data/datapacks): Australia, SA2, General Community Profile, G02.
- [2021 allocation files](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/allocation-files): mesh block → SA2/state and mesh block → Postal Area (POA).
- [2021 mesh-block counts](https://www.abs.gov.au/census/guide-census-data/mesh-block-counts/2021): usual-resident population for the weights.

Read the G02 table (selected medians and averages) from the Census ZIP for SA2 income. Preserve geographic codes as strings to retain leading zeros. All income data and geographic boundaries use 2021 data, as the census is collected every 5 years. The next iteration after 2021 is in 2026. Save the ABS extracts as Parquet in landing; reuse existing extracts on reruns.

In [20]:
# Set up external dataset access URLs
allocation_url = ("https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/"
                  "edition-3-july-2021-june-2026/access-and-downloads/allocation-files/")
abs_urls = {
    "sa2_income": "https://www.abs.gov.au/census/find-census-data/datapacks/download/2021_GCP_SA2_for_AUS_short-header.zip",
    "mb_sa2": allocation_url + "MB_2021_AUST.xlsx",
    "mb_poa": allocation_url + "POA_2021_AUST.xlsx",
    "mb_population": "https://www.abs.gov.au/census/guide-census-data/mesh-block-counts/2021/Mesh%20Block%20Counts%2C%202021.xlsx",
}
ABS.mkdir(parents=True, exist_ok=True)

# Downloads live in a temporary directory; only Parquet source extracts are retained.
with TemporaryDirectory(prefix="bnpl_abs_") as temp:
    for name, url in abs_urls.items():
        target = ABS / name / "part-00000.parquet"
        # Reuse completed extracts so reruns avoid downloading the same ABS files.
        if target.exists():
            continue
        source = Path(temp) / (name + (".zip" if name == "sa2_income" else ".xlsx"))
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        source.write_bytes(response.content)
        # Read only G02 from the Census ZIP, each dataset differs in structure and needs to be read in differently
        if name == "sa2_income":
            with ZipFile(source) as archive:
                member = next(n for n in archive.namelist() if n.endswith("G02_AUST_SA2.csv"))
                frame = pd.read_csv(archive.open(member), dtype=str)
        elif name == "mb_population":
            # Stack the population tables, skipping workbook headings and retaining valid mesh-block rows.
            with pd.ExcelFile(source) as workbook:
                sheets = [s for s in workbook.sheet_names if s.startswith("Table ")]
                frame = pd.concat([pd.read_excel(workbook, sheet_name=s, skiprows=6,
                                   usecols=["MB_CODE_2021", "Person"], dtype=str) for s in sheets], ignore_index=True)
            frame = frame[frame.MB_CODE_2021.str.fullmatch(r"\d{11}", na=False)]
        else:
            columns = (["MB_CODE_2021", "SA2_CODE_2021", "STATE_CODE_2021"] if name == "mb_sa2"
                       else ["MB_CODE_2021", "POA_CODE_2021"])
            frame = pd.read_excel(source, usecols=columns, dtype=str)
        target.parent.mkdir(parents=True, exist_ok=True)
        frame.to_parquet(target, index=False, compression=None)

## 3. Raw: read the supplied datasets
We perform an initial parse of the following data to raw layer. Keep source names and values where possible.
| Dataset | table name | Original format |
|---|---|---|
| Consumers | `tbl_consumer.csv` | CSV, pipe-delimited (`\|`) |
| Consumer fraud | `consumer_fraud_probability.csv` | CSV, comma-delimited |
| Merchant fraud | `merchant_fraud_probability.csv` | CSV, comma-delimited |
| Users | `consumer_user_details.parquet` | Parquet |
| Merchants | `tbl_merchants.parquet` | Parquet |
| Transactions | `transactions*_snapshot/` | Parquet files, partitioned by `order_datetime` |


A few things to note:
- for `merchant` table, it stores merchant ABN as index, but spark reads as column directly, so pandas `reset_index()` as performed in `00_preliminary_analysis.ipynb` is unnecessary here
- for `consumers` and `fraud` CSVs, we explicit set schemas prevent IDs/postcodes being inferred incorrectly in spark reading using `spark.read.schema(schema).csv(...)`
- for `transaction` snapshots, add `order_datetime` column (recovered from partition folder name) and combine all partitions

Schema is specified using these general rules following the column profiling from `00_preliminary_analysis.ipynb`: 
- text or categories (e.g. `name`, `address`, `gender`, `state`) $\rightarrow$ string 
- geographical information (e.g. `postcode`) $\rightarrow$ string 
- record identifiers (e.g. `consumer_id`, `user_id`, `merchant_abn`) $\rightarrow$ long 
- numeric scores (e.g. `fraud_probability`) $\rightarrow$ double 

In [21]:
def read_csv(name, schema, sep=","):
    """Read a Part 1 CSV into Spark using the supplied schema and delimiter. Stops read when malformed record encountered (FAILFAST)"""
    return (spark.read.option("header", True).option("sep", sep).option("mode", "FAILFAST") # if full parse complete, no malformed records
            .option("enforceSchema", False).schema(schema).csv(str(PART1 / name)))

# Read CSVs using specified schema and read parquets directly, to a dictionary holding separate spark dfs
raw = {
    "consumers": read_csv("tbl_consumer.csv", "name string, address string, state string, postcode string, gender string, consumer_id long", "|"),
    "users": spark.read.parquet(str(PART1 / "consumer_user_details.parquet")),
    "merchants": spark.read.parquet(str(PART1 / "tbl_merchants.parquet")),
    "consumer_fraud": read_csv("consumer_fraud_probability.csv", "user_id long, order_datetime date, fraud_probability double"),
    "merchant_fraud": read_csv("merchant_fraud_probability.csv", "merchant_abn long, order_datetime date, fraud_probability double"),
}
snapshots = sorted(LANDING.glob("project-2-bnpl-tables-part*/transactions*_snapshot"))
if not snapshots:
    raise FileNotFoundError("No transaction snapshots found in landing.")

# Read snapshot roots to get dates from partition folder names, then convert to a column with date type
transactions = [(spark.read.option("basePath", str(p)).parquet(str(p))
                 .withColumn("order_datetime", F.to_date("order_datetime"))
                 .withColumn("source_snapshot", F.lit(p.name))) for p in snapshots]

# Stack snapshots by matching column names
raw["transactions"] = transactions[0]
for snapshot in transactions[1:]:
    raw["transactions"] = raw["transactions"].unionByName(snapshot)

## 4. Raw: standardise ABS keys and save Parquet tables

The six supplied datasets from part 3 each get their own directory. ABS lookups get four additional directories. Raw transactions are partitioned by `order_month`. For downstream analysis and modelling, we will read from raw/curated layer saved outputs. 

Here, we will perform column name standardisation (especially for keys) to enable table joining later. This is the last step for raw layer.

All supplied datasets read and standardised are collected in a python dictionary `raw` as spark dataframes.

In [22]:
# Give geographic keys consistent names for joins, then add to dictionary of spark dfs as in part 3
raw["sa2_income"] = spark.read.parquet(str(ABS / "sa2_income")).select(
    F.col("SA2_CODE_2021").alias("sa2_code"),
    F.col("Median_tot_prsnl_inc_weekly").cast("double").alias("median_personal_income_weekly")) # cast to numeric
raw["mb_sa2"] = spark.read.parquet(str(ABS / "mb_sa2")).select(
    F.col("MB_CODE_2021").alias("mb_code"), F.col("SA2_CODE_2021").alias("sa2_code"),
    F.col("STATE_CODE_2021").alias("state_code"))
raw["mb_poa"] = spark.read.parquet(str(ABS / "mb_poa")).select(
    F.col("MB_CODE_2021").alias("mb_code"), F.col("POA_CODE_2021").alias("postcode"))
raw["mb_population"] = spark.read.parquet(str(ABS / "mb_population")).select(
    F.col("MB_CODE_2021").alias("mb_code"), F.col("Person").cast("long").alias("population")) # cast to numeric

# Add `order_month` for partitioning saved parquet into monthly folders
raw["transactions"] = raw["transactions"].withColumn("order_month", F.date_format("order_datetime", "yyyy-MM"))

# Save raw tables to disk as raw layer (partition by month if transactions)
for name, df in raw.items():
    write_table(df, RAW / name, "order_month" if name == "transactions" else None)

# Reload the saved raw tables so curated processing starts from a reproducible on-disk boundary
raw = {name: spark.read.parquet(str(RAW / name)) for name in raw}

## 5. Curated: replace the user bridge and prepare merchant/fraud tables

For the curated layer, we will first try to simplify joins and also deal with some join inconsistencies: 
1. For `consumers` and `users`: merge consumer details onto the user mappings and drop `consumer_id` such that the curated consumer key is now **`user_id`** used for all downstream analysis and modelling 
2. For `consumer`: zero-pad postcodes to valid 4-digit Australian postcodes
3. For `merchants`: parse merchant `tags` into category, letter group and take rate. Keep take rates and fraud scores in their supplied units for now. 
4. For `fraud`: remove exact duplicate score rows, throw error if any conflicting scores for a single key = (consumer/merchant id, order datetime)

In [23]:
# Validate user-consumer mapping and one-row-per-order assumption using defined `require_key` function
for name, keys in [("consumers", ["consumer_id"]), ("users", ["user_id"]), ("users", ["consumer_id"]),
                   ("merchants", ["merchant_abn"]), ("transactions", ["order_id"])]:
    require_key(raw[name], keys, name)

# Users table contains `user_id` which joins with all other tables
consumers = (raw["users"].join(raw["consumers"].withColumn("consumer_record_matched", F.lit(True)), "consumer_id", "left") # flags missing details
             .drop("consumer_id").fillna(False, subset=["consumer_record_matched"])
             .withColumn("state", F.upper(F.trim("state")))) # standardise state text

# Confirm every user has a matching consumer record
unmatched = consumers.filter(~F.col("consumer_record_matched")).count()
print(f"Users without consumer details: {unmatched:,}")

# Confirm no consumers were left out of the user-based curated table
unmapped = raw["consumers"].join(raw["users"], "consumer_id", "left_anti").count()
print(f"Consumers without a user mapping: {unmapped:,}")

Users without consumer details: 0
Consumers without a user mapping: 0


In [24]:
# For consumer table, 0-pad postcodes in consumers table for valid Australian postcode
consumers = consumers.withColumn("postcode", F.when(F.trim("postcode").rlike(r"^\d{3,4}$"),
                                 F.lpad(F.trim("postcode"), 4, "0")).otherwise(F.col("postcode")))

# For mechant table, parse tag into category / letter / take-rate fields
pattern = r"(?i)^[\[(]{2}(.*?)[\])]\s*,\s*[\[(]([A-Za-z])[\])]\s*,\s*[\[(]take rate:\s*([0-9]+(?:\.[0-9]+)?)[\])]{2}$" # regex pattern to extract each tag component
merchants = (raw["merchants"]
    .withColumn("category", F.lower(F.trim(F.regexp_extract("tags", pattern, 1))))
    .withColumn("letter_group", F.lower(F.regexp_extract("tags", pattern, 2)))
    .withColumn("take_rate", F.when(F.regexp_extract("tags", pattern, 3) != "", F.regexp_extract("tags", pattern, 3).cast("double"))))

# Match the preliminary category rule: collapse whitespace and standardise comma/colon/hyphen spacing.
merchants = merchants.withColumn("category", F.trim(F.regexp_replace(
    F.regexp_replace(F.regexp_replace("category", r"\s+", " "), r"\s*([,:])\s*", "$1 "), r"\s*-\s*", " - ")))

# For merchant table, sanity check for tag parse
if merchants.filter(F.col("take_rate").isNull()).limit(1).count():
    raise ValueError("Some merchant tags did not parse; inspect them before continuing.")

# For fraud tables, remove identical score rows
consumer_fraud = raw["consumer_fraud"].dropDuplicates().withColumnRenamed("fraud_probability", "consumer_fraud_probability")
merchant_fraud = raw["merchant_fraud"].dropDuplicates().withColumnRenamed("fraud_probability", "merchant_fraud_probability")
require_key(consumer_fraud, ["user_id", "order_datetime"], "consumer fraud")

# For fraud tables, fail if conflicting scores for same key = (consumer/merchant, datetime))
require_key(merchant_fraud, ["merchant_abn", "order_datetime"], "merchant fraud")

## 6. Curated: postcode-SA2 median income lookup

`Consumers` table have state/postcode, not SA2. [ABS Postal Areas](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/non-abs-structures/postal-areas) approximate postcodes and may cross state borders, which we will adjust using consumer given state. We will use mesh blocks (smallest ABS geographical unit) to establish state-postcode overlap with SA2, weighting each overlap by its usual-resident population. This is a derived mapping using public ABS data.

For each state/postcode area $p$, we calculate the estimated (median) weekly person income `district_income`: 
$$\widehat{I}_p
=
\frac{\displaystyle\sum_{s \in S_p} N_{ps}\,m_s}
     {\displaystyle\sum_{s \in S_p} N_{ps}}$$
- $S_p$: SA2 areas overlapping postcode area $p$ with usable income data
- $N_{ps}$: population living in the overlap between postcode area $p$ and SA2 $s$
- $m_s$: median weekly personal income in SA2 $s$, in dollars
- $\widehat{I}_p$: population-weighted average of those SA2 medians. NULL if population is zero. 

This is an **average of SA2 medians**, to be joined with individual consumers, NOT the postcode median! Population is an approximated geographic weight, as personal income is for people aged 15+ only. Exclude non-geographic ABS codes from the income lookup such as `9494` (no usual address), `9797` (migratory/offshore/shipping) and `ZZZZ` (outside Australia).

Create table with calculated (weighted) median income per state/postcode to join to consumers. Note the following join assumptions:
- For each postcode, median income left as NULL if all its mapped SA2 zones have NULL income or zero total population 
     - as justified in `00_preliminary_analysis.ipynb`, we will also exclude the 42 SA2 zones with 0-income due to data ambiguity (and would skew the data if averaged)
- For each postcode, record number of mapped SA2 zones and observed population
- Assignment specifies SA2 datasets from ABS, otherwise we might consider using direct POA income table...

The created dataframes are of the following structure:
| Table | Key | Contains |
|---|---|---|
| `mesh` | `mb_code` | Population per mesh block |
| `overlap` | `state`, `postcode`, `sa2_code` | Population per state–POA–SA2 overlap |
| `overlap_income` | `state`, `postcode`, `sa2_code` | Overlap population and SA2 median income |
| `postcode_income` | `state`, `postcode` | Population and weighted income estimate per state–POA |

For `postcode_income`, the columns are: 
- `income_sa2_count`: number of sa2 zones overlapping with this state-poa
- `area_population`: population per state-poa (all)
- `income_population`: population per state-poa with nonzero median income
- `district_income`: weighted median income avg per state-poa
- `income_coverage`: proportion of state-poa with nonzero median income

In [25]:
# Check ABS data tables contain required keys
for name, keys in [("sa2_income", ["sa2_code"]), ("mb_sa2", ["mb_code"]),
                   ("mb_poa", ["mb_code"]), ("mb_population", ["mb_code"])]:
    require_key(raw[name], keys, name) 

# State code mapping between ABS `state_code_2021` and consumers `state`
state_codes = {"1": "NSW", "2": "VIC", "3": "QLD", "4": "SA", "5": "WA", "6": "TAS", "7": "NT", "8": "ACT", "9": "OT"}
state_map = F.create_map(*[F.lit(v) for pair in state_codes.items() for v in pair])

# Mesh blocks dataframe: one mesh per row, with SA2, postcode, population, state abbreviation   
mesh = (raw["mb_sa2"].join(raw["mb_poa"], "mb_code", "left")
        .join(raw["mb_population"], "mb_code", "left")
        .withColumn("state", state_map[F.col("state_code")]))

# Identify all SA2s overlapping each state/postcode and sums their overlap population
overlap = (mesh.filter(F.col("postcode").rlike(r"^\d{4}$")       # exclude irregular format codes
                       & ~F.col("postcode").isin("9494", "9797") # exclude non-graphic POA codes
                       & (F.col("population") > 0))              # exclude unpopulated blocks 
           .groupBy("state", "postcode", "sa2_code").agg(F.sum("population").alias("population"))) # population per (state)-poa-sa2

# Validate each row with (state)-poa-sa2 key
require_key(overlap, ["state", "postcode", "sa2_code"], "postcode-SA2 overlaps")
write_table(overlap, CURATED / "postcode_sa2_population")

# Attach per-sa2 income to overlap (state-poa-sa2) regions
overlap_income = overlap.join(raw["sa2_income"], "sa2_code", "left")

# Calculate weighted average per state-poa: weighted income sum divided by population with usable income
postcode_income = (overlap_income.groupBy("state", "postcode").agg( # new key
    F.countDistinct("sa2_code").alias("income_sa2_count"), 
    F.sum("population").alias("area_population"), # postcode population
    F.sum(F.when(F.col("median_personal_income_weekly") > 0, 
                 F.col("population")).otherwise(0)).alias("income_population"), # state-poa population, summed from state-poa-sa2 populations with positive median income
    F.sum(F.when(F.col("median_personal_income_weekly") > 0, 
                 F.col("population") * F.col("median_personal_income_weekly"))).alias("income_sum")) # weighted income sum per state-poa when income is postive
    .withColumn("district_income", F.when(F.col("income_population") > 0, 
                                          F.col("income_sum") / F.col("income_population"))) # weighted income avg per state-poa when usable population is non-zero
    .withColumn("income_coverage", F.col("income_population") / F.col("area_population")) # proportion of state-poa with positive income 
    .drop("income_sum")) # drop intermediate column

# Validate each row with state-poa key
require_key(postcode_income, ["state", "postcode"], "postcode income")
write_table(postcode_income, CURATED / "postcode_income")

## 7. Curated: join calculated median income to consumers

Left-join calculated median income lookup table on **state + postcode**, preserving every user mapping. Missing income stays null. 
- `income_matched`: TRUE if an estimate is available
- `income_coverage`: whether the weighted median income is calculated from all resident population in the state/POA overlap

In [26]:
# Read from postcode income lookup
postcode_income = spark.read.parquet(str(CURATED / "postcode_income"))

# Broadcast the small postcode lookup to each partition to avoid shuffle
income_lookup = postcode_income.drop("area_population", "income_population")

# Remove previous enrichment columns on rerun
income_columns = [c for c in income_lookup.columns if c not in ["state", "postcode"]]
consumers = consumers.drop(*income_columns, "income_matched")
consumers = (consumers.join(F.broadcast(income_lookup), ["state", "postcode"], "left") # left join keeps users with no income match
             .withColumn("income_matched", F.col("district_income").isNotNull())) # flag if no match

# Validate each row with consumer key 
require_key(consumers, ["user_id"], "curated consumers")

# Save curated layer reference tables
curated = {"consumers": consumers, "merchants": merchants, "consumer_fraud": consumer_fraud, "merchant_fraud": merchant_fraud}
for name, df in curated.items():
    write_table(df, CURATED / name)
curated = {name: spark.read.parquet(str(CURATED / name)) for name in curated}

## 8. Curated: enrich transactions

Keep **one row per order** with the diagram's five additional attributes: consumer/merchant fraud probabilities, consumer postcode, gender and district income. Merchant name/tags stay in the separate curated merchant table, linked by ABN.

Joining logic: 
- Join entity fraud scores directly on **entity ID + order date**, independently of whether entity details exist. Entity is either consumer or merchant.
- Left joins keep unmatched orders and null scores.
- `merchant_record_matched` and `consumer_record_matched` distinguish missing master records. 
- Keep income coverage/year alongside its estimate. 

In [27]:
# Only select consumer attributes reuqired for each order
consumer_attributes = curated["consumers"].select(
    "user_id", 
    F.col("postcode").alias("consumer_postcode"), 
    F.col("gender").alias("consumer_gender"),
    F.col("district_income").alias("consumer_district_income"), 
    "consumer_record_matched",
    "income_matched", 
    "income_coverage")

# Check if merchant exists in merchant table (consumers checked earlier in section 5)
merchant_keys = curated["merchants"].select("merchant_abn").withColumn("merchant_record_matched", F.lit(True))

# Join required columns (merchant and fraud score lookups are broadcasted)
transactions = (raw["transactions"].join(consumer_attributes, "user_id", "left") 
                .join(F.broadcast(merchant_keys), "merchant_abn", "left")
                .join(F.broadcast(curated["consumer_fraud"]), ["user_id", "order_datetime"], "left") # requires exact entity-date match
                .join(F.broadcast(curated["merchant_fraud"]), ["merchant_abn", "order_datetime"], "left")
                .fillna(False, subset=["consumer_record_matched", "merchant_record_matched", "income_matched"])) # flag false if unknown score

# Select necessary columns and save to disk
transactions = transactions.select("order_id", "order_datetime", "order_month", "dollar_value", 
                                   "user_id", "merchant_abn", "consumer_fraud_probability", "merchant_fraud_probability",
                                   "consumer_postcode", "consumer_gender", "consumer_district_income", 
                                   "income_matched", "consumer_record_matched", "merchant_record_matched")

write_table(transactions, CURATED / "transactions", "order_month")

## 9. Verify row preservation

Check the saved result against raw that all rows have been preserved appropriately.

In [28]:
# Compare persisted outputs to catch any rows lost or multiplied by the enrichment joins
raw_count = raw["transactions"].count()
curated_count = spark.read.parquet(str(CURATED / "transactions")).count()
if curated_count != raw_count:
    raise ValueError(f"Transaction count changed: {raw_count:,} raw → {curated_count:,} curated.")
print(f"Saved raw and curated Parquet tables; retained {curated_count:,} transactions.")

Saved raw and curated Parquet tables; retained 14,195,505 transactions.


## 10. Some `transactions` statistics
Check the data in `transations`, especially orders with missing important attributes.

In [29]:
transaction_total = transactions.count()
transaction_unknown_consumer_fraud = transactions.where(F.col('consumer_fraud_probability').isNull()).count()
transaction_unknown_merchant_fraud = transactions.where(F.col('merchant_fraud_probability').isNull()).count()
transaction_unknown_or_fraud = transactions.where((F.col('consumer_fraud_probability').isNull()) | (F.col('merchant_fraud_probability').isNull())).count()
transaction_unknown_and_fraud = transactions.where((F.col('consumer_fraud_probability').isNull()) & (F.col('merchant_fraud_probability').isNull())).count()

print(f"# Orders in total: {transaction_total}")
print(f"# Orders missing consumer_fraud probability: {transaction_unknown_consumer_fraud} ({transaction_unknown_consumer_fraud/transaction_total:.4%})")
print(f"# Orders missing merchant_fraud probability: {transaction_unknown_merchant_fraud} ({transaction_unknown_merchant_fraud/transaction_total:.4%})")
print(f"# Orders missing either fraud probability: {transaction_unknown_or_fraud} ({transaction_unknown_or_fraud/transaction_total:.4%})")
print(f"# Orders missing both fraud probability: {transaction_unknown_and_fraud} ({transaction_unknown_and_fraud/transaction_total:.4%})")

# Orders in total: 14195505
# Orders missing consumer_fraud probability: 14115157 (99.4340%)
# Orders missing merchant_fraud probability: 14191446 (99.9714%)
# Orders missing either fraud probability: 14195100 (99.9971%)
# Orders missing both fraud probability: 14111503 (99.4082%)


In [30]:
print(transactions.where(F.col("order_id").isNull()).count())
print(transactions.where(F.col("order_datetime").isNull()).count())
print(transactions.where(F.col("order_month").isNull()).count())
print(transactions.where(F.col("dollar_value").isNull()).count())
print(transactions.where(F.col("user_id").isNull()).count())
print(transactions.where(F.col("merchant_abn").isNull()).count())
print(transactions.where(F.col("consumer_fraud_probability").isNull()).count())
print(transactions.where(F.col("merchant_fraud_probability").isNull()).count())
print(transactions.where(F.col("consumer_postcode").isNull()).count())
print(transactions.where(F.col("consumer_gender").isNull()).count())
print(transactions.where(F.col("consumer_district_income").isNull()).count())

0
0
0
0
0
0
14115157
14191446
0
0
2368565


In [31]:
transactions.select(
    F.min("consumer_district_income"), 
    F.max("consumer_district_income")
    ).show()

+-----------------------------+-----------------------------+
|min(consumer_district_income)|max(consumer_district_income)|
+-----------------------------+-----------------------------+
|                        276.0|                       3250.0|
+-----------------------------+-----------------------------+



## Review before treating this as the final pipeline

1. **Income definition/mapping:** median weekly personal income, population-weighted SA2 proxy by state/postcode. Review the all-age weights, zero-income rule, partial coverage and POA approximation. We have not inferred an individual's SA2 or income.
2. **Transaction joins:** currently preserve all orders and attach every available entity/date score, even when a merchant master record is absent. Decide separately whether ranking should exclude unknown merchants. Sparse fraud coverage remains null.
3. **Future data:** include every actual supplied partition; stop on repeated order IDs or conflicting fraud keys. Future overlapping releases need an explicit precedence rule. Confirm take-rate/score units before modelling; no fraud threshold or outlier removal is applied here.
4. **Historical modelling:** 2021 Census data were published from June 2022. This is retrospective enrichment; do not treat it as information available before release in a historical prediction exercise.

Expected curated directories: `consumers`, `merchants`, `consumer_fraud`, `merchant_fraud`, `postcode_sa2_population`, `postcode_income`, `transactions`. There is **no curated Users table**; raw retains it for traceability.

In [32]:
import shutil
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tables/curated/transactions").is_dir())
transactions_path = root / "tables/curated/transactions"
temporary_path = root / "tables/curated/transactions_enriched_temp"
backup_path = root / "tables/curated/transactions_before_enrichment"

transactions_to_enrich = spark.read.parquet(str(transactions_path))
median_district_income = transactions_to_enrich.approxQuantile("consumer_district_income", [0.5], 0.001)[0]
consumer_window = Window.partitionBy("user_id")

enriched_transactions = (transactions_to_enrich
    .withColumn("consumer_district_income",
                F.coalesce(F.col("consumer_district_income"), F.lit(median_district_income)))
    .withColumn("consumer_avg_spending", F.avg("dollar_value").over(consumer_window))
    .withColumn("spending_diff", F.abs(F.col("dollar_value") - F.col("consumer_avg_spending"))))

if temporary_path.exists():
    shutil.rmtree(temporary_path)
if backup_path.exists():
    shutil.rmtree(backup_path)

enriched_transactions.write.mode("overwrite").partitionBy("order_month").parquet(str(temporary_path))
transactions_path.rename(backup_path)
temporary_path.rename(transactions_path)
shutil.rmtree(backup_path)

In [33]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

# consumer_avg_spending = historical avg transaction not including current

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tables/curated/transactions").is_dir())
spark = SparkSession.builder.getOrCreate()
transactions_history = spark.read.parquet(str(root / "tables/curated/transactions"))
historical_window = (Window.partitionBy("user_id")
    .orderBy("order_datetime", "order_id")
    .rowsBetween(Window.unboundedPreceding, -1))
transactions_history = transactions_history.withColumn(
    "consumer_avg_spending", F.avg("dollar_value").over(historical_window)
)
transactions_history.orderBy("user_id", "order_datetime", "order_id").show(5, truncate=False)

+------------------------------------+--------------+------------------+-------+------------+--------------------------+--------------------------+-----------------+---------------+------------------------+--------------+-----------------------+-----------------------+---------------------+------------------+-----------+
|order_id                            |order_datetime|dollar_value      |user_id|merchant_abn|consumer_fraud_probability|merchant_fraud_probability|consumer_postcode|consumer_gender|consumer_district_income|income_matched|consumer_record_matched|merchant_record_matched|consumer_avg_spending|spending_diff     |order_month|
+------------------------------------+--------------+------------------+-------+------------+--------------------------+--------------------------+-----------------+---------------+------------------------+--------------+-----------------------+-----------------------+---------------------+------------------+-----------+
|0c37b3f7-c7f1-48cb-bcc7-0a58e7

In [34]:
transactions_df = spark.read.parquet(str(root / "tables/curated/transactions"))
null_counts_df = transactions_df.select([
    F.sum(F.col(column).isNull().cast("long")).alias(column)
    for column in transactions_df.columns
])
null_counts_df.show(truncate=False)

+--------+--------------+------------+-------+------------+--------------------------+--------------------------+-----------------+---------------+------------------------+--------------+-----------------------+-----------------------+---------------------+-------------+-----------+
|order_id|order_datetime|dollar_value|user_id|merchant_abn|consumer_fraud_probability|merchant_fraud_probability|consumer_postcode|consumer_gender|consumer_district_income|income_matched|consumer_record_matched|merchant_record_matched|consumer_avg_spending|spending_diff|order_month|
+--------+--------------+------------+-------+------------+--------------------------+--------------------------+-----------------+---------------+------------------------+--------------+-----------------------+-----------------------+---------------------+-------------+-----------+
|0       |0             |0           |0      |0           |14115157                  |14191446                  |0                |0              |0

In [35]:
import shutil
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tables/curated/merchants").is_dir())
transactions_path = root / "tables/curated/transactions"
merchants_path = root / "tables/curated/merchants"
temporary_path = root / "tables/curated/merchants_enriched_temp"
backup_path = root / "tables/curated/merchants_before_enrichment"

transactions_for_merchants = spark.read.parquet(str(transactions_path))
merchant_metrics = (transactions_for_merchants.groupBy("merchant_abn").agg(
    F.avg("merchant_fraud_probability").alias("avg_merchant_fraud_prob"),
    F.avg("consumer_district_income").alias("avg_consumer_district_income"),
    F.avg("consumer_fraud_probability").alias("avg_consumer_fraud_prob"),
    F.sum("dollar_value").alias("total_revenue"),
    F.count(F.lit(1)).alias("total_transactions"))
    .withColumn("avg_value_per_transaction", F.col("total_revenue") / F.col("total_transactions")))

metric_columns = ["avg_merchant_fraud_prob", "avg_consumer_district_income",
                  "avg_consumer_fraud_prob", "total_revenue", "total_transactions",
                  "avg_value_per_transaction", "expected_revenue"]
merchants_to_enrich = spark.read.parquet(str(merchants_path)).drop(*metric_columns)
enriched_merchants = (merchants_to_enrich.join(merchant_metrics, "merchant_abn", "left")
    .withColumn("expected_revenue",
                F.col("take_rate") * F.col("total_revenue") * F.col("avg_consumer_fraud_prob")))

if temporary_path.exists():
    shutil.rmtree(temporary_path)
if backup_path.exists():
    shutil.rmtree(backup_path)

enriched_merchants.write.mode("overwrite").parquet(str(temporary_path))
merchants_path.rename(backup_path)
temporary_path.rename(merchants_path)
shutil.rmtree(backup_path)

In [36]:
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tables/curated/merchants").is_dir())
transaction_file = next((root / "tables/curated/merchants").rglob("*.parquet"))
spark = SparkSession.builder.getOrCreate()
spark.read.parquet(str(transaction_file)).show(5, truncate=False)

+------------+--------------------------+-----------------------------------------------------------------------------------------+-------------------------------------------------------------+------------+---------+-----------------------+----------------------------+-----------------------+------------------+------------------+-------------------------+-------------------+
|merchant_abn|name                      |tags                                                                                     |category                                                     |letter_group|take_rate|avg_merchant_fraud_prob|avg_consumer_district_income|avg_consumer_fraud_prob|total_revenue     |total_transactions|avg_value_per_transaction|expected_revenue   |
+------------+--------------------------+-----------------------------------------------------------------------------------------+-------------------------------------------------------------+------------+---------+-----------------------+----

26/09/11 15:21:10 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo